# Transisi Energi Hijau di Indonesia
**Analisis Pemberitaan Media Nasional**

**Periode:** 17 April – 17 Mei 2026  
**Sumber:** Semantik API (keyword-filtered) — 72 artikel dari 12 media nasional  
**Kata kunci:** transisi energi, energi hijau, EBT, energi baru terbarukan  

> **Metodologi:** Menggunakan keyword-filtered endpoints dari Semantik API — hanya artikel yang mengandung kata kunci spesifik transisi energi hijau. Bukan channel-level (yang mencakup seluruh isu lingkungan).

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['figure.figsize'] = (10, 5)

DATA_DIR = 'data'
CHARTS_DIR = 'charts'

def load_json(filename):
    with open(os.path.join(DATA_DIR, filename)) as f:
        return json.load(f)

print('Setup complete. Data files:', os.listdir(DATA_DIR))

## 1. Volume & Sumber Artikel

72 artikel dari 12 media nasional selama periode 17 April – 17 Mei 2026.

In [ ]:
source_data = load_json('source_comparison.json')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: articles per source
sources = [s['source'] for s in source_data]
counts = [s['article_count'] for s in source_data]
colors = plt.cm.viridis([i/len(sources) for i in range(len(sources))])

axes[0].barh(sources, counts, color=colors)
axes[0].set_xlabel('Jumlah Artikel')
axes[0].set_title('Artikel per Media')
axes[0].invert_yaxis()

# Sentiment distribution per source
pos = [s['sentiment_distribution']['positive'] for s in source_data]
neg = [s['sentiment_distribution']['negative'] for s in source_data]
neu = [s['sentiment_distribution']['neutral'] for s in source_data]

axes[1].barh(sources, pos, label='Positif', color='#2ecc71')
axes[1].barh(sources, neg, left=pos, label='Negatif', color='#e74c3c')
axes[1].barh(sources, neu, left=[p+n for p,n in zip(pos,neg)], label='Netral', color='#95a5a6')
axes[1].set_xlabel('Jumlah Artikel')
axes[1].set_title('Distribusi Sentimen per Media')
axes[1].invert_yaxis()
axes[1].legend()

plt.tight_layout()
plt.show()

### Volume Mingguan

In [ ]:
trend_data = load_json('topic_trend.json')

# Group by week
from datetime import datetime
from collections import defaultdict

weekly = defaultdict(int)
for d in trend_data:
    dt = datetime.strptime(d['date'], '%Y-%m-%d')
    week = dt.isocalendar()[1]
    weekly[week] += d['article_count']

weeks = sorted(weekly.keys())
week_labels = [f'W{w}' for w in weeks]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([f'W{w}' for w in weeks], [weekly[w] for w in weeks], color='#3498db')
for bar, val in zip(bars, [weekly[w] for w in weeks]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(val), ha='center', fontweight='bold')
ax.set_ylabel('Jumlah Artikel')
ax.set_title('Volume Artikel per Minggu')
plt.tight_layout()
plt.show()

## 2. Sentimen Entitas Utama

Tiga entitas utama yang dianalisis: **Pertamina**, **PLN**, dan **Prabowo**.

In [ ]:
entities = ['pertamina', 'pln', 'prabowo']
sentiments = {e: load_json(f'{e}_sentiment.json') for e in entities}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (entity, data) in enumerate(sentiments.items()):
    labels = ['Positif', 'Negatif', 'Netral']
    values = [data['positive'], data['negative'], data.get('neutral', 0)]
    colors = ['#2ecc71', '#e74c3c', '#95a5a6']
    
    wedges, texts, autotexts = axes[i].pie(values, labels=labels, colors=colors, autopct='%1.0f%%', startangle=90)
    axes[i].set_title(f"{entity.upper()}\nSkor: {data['average_score']:+.2f} | {data['article_count']} sebutan")

plt.suptitle('Distribusi Sentimen per Entitas', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Pertamina: {sentiments['pertamina']['average_score']:+.2f} (positif tapi framing BBM, bukan hijau)")
print(f"PLN: {sentiments['pln']['average_score']:+.2f} (negatif karena pemadaman Jakarta)")
print(f"Prabowo: {sentiments['prabowo']['average_score']:+.2f} (positif dari pengumuman PLTS 100 GW)")

## 3. Timeline Sebutan Harian

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for i, entity in enumerate(entities):
    timeline = load_json(f'{entity}_timeline.json')
    dates = [d['date'] for d in timeline]
    counts = [d['mention_count'] for d in timeline]
    
    axes[i].fill_between(range(len(dates)), counts, alpha=0.3, color=['#f39c12', '#3498db', '#9b59b6'][i])
    axes[i].plot(range(len(dates)), counts, color=['#f39c12', '#3498db', '#9b59b6'][i], linewidth=1.5)
    axes[i].set_ylabel('Sebutan/hari')
    axes[i].set_title(entity.upper())
    
    # Annotate peaks
    max_idx = counts.index(max(counts))
    axes[i].annotate(f'{dates[max_idx]}: {counts[max_idx]}', xy=(max_idx, counts[max_idx]),
                     xytext=(max_idx+2, counts[max_idx]+1), arrowprops=dict(arrowstyle='->', color='gray'),
                     fontsize=9, fontweight='bold')

# Set x-axis labels (every 5th date)
tick_pos = list(range(0, len(dates), 5))
axes[-1].set_xticks(tick_pos)
axes[-1].set_xticklabels([dates[i] for i in tick_pos], rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 4. Ko-okurensi Entitas

Analisis entitas mana yang sering muncul bersamaan — menunjukkan konteks pemberitaan.

In [ ]:
cooc_entities = ['pertamina', 'nikel', 'pln', 'prabowo']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, entity in enumerate(cooc_entities):
    ax = axes[idx//2][idx%2]
    data = load_json(f'{entity}_cooccurrence.json')
    
    # Top 8 co-occurring entities
    coocs = data['co_occurring_entities'][:8]
    labels = [c['word'] for c in coocs]
    values = [c['co_occurrence_count'] for c in coocs]
    
    colors = plt.cm.RdYlGn([v/max(values) for v in values])
    ax.barh(labels, values, color=colors)
    ax.set_title(f'{entity.upper()} — Ko-okurensi ({data["mention_count"]} sebutan)')
    ax.invert_yaxis()
    
    for j, v in enumerate(values):
        ax.text(v + 0.3, j, str(v), va='center', fontsize=9)

plt.suptitle('Ko-okurensi Entitas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Framing Analisis

Bagaimana media membingkai setiap entitas — apakah dalam konteks transisi energi hijau?

In [ ]:
for entity in ['pertamina', 'pln', 'prabowo']:
    framing = load_json(f'{entity}_framing.json')
    print(f"\n{'='*60}")
    print(f"FRAMING: {entity.upper()} ({framing['entity_word']})")
    print('='*60)
    
    for source, frames in list(framing['by_source'].items())[:6]:
        print(f"\n📰 {source}:")
        for frame in frames[:3]:
            phrase = frame.get('framing_phrase', frame.get('frame', 'N/A'))
            count = frame.get('article_count', 0)
            print(f"  • {phrase} ({count} artikel)")

## 6. Jaringan Relasi

In [ ]:
network = load_json('network.json')
print(f"Total nodes: {len(network['nodes'])}")
print(f"Total edges: {network['total_edges']}")

# Show top edges
edges = sorted(network['edges'], key=lambda e: e.get('weight', 0), reverse=True)[:15]
print("\nTop 15 relasi:")
for e in edges:
    print(f"  {e['source']} → {e['target']} (weight: {e.get('weight', '?')})")

## 7. Temuan Kunci

### V1 vs V2

| Metrik | V1 (Channel) | V2 (Keyword) |
|--------|:--:|:--:|
| Total Artikel | 2.264 | 72 |
| Sumber Media | 14 | 12 |
| Metode | `channel=Lingkungan` | `topic_keywords=...` |
| Cakupan | Seluruh isu lingkungan | Spesifik transisi energi |

V1 menggunakan channel-level Lingkungan (2.264 artikel). Hanya **72 (3%)** yang benar-benar relevan.

### Temuan Analisis

1. **Volume transisi energi hijau sangat rendah** — 72 artikel/bulan dari 12 media nasional. Transisi energi belum jadi prioritas pemberitaan.

2. **Pertamina dominan tapi tidak hijau** — entitas paling disebut (307 sebutan), tapi framing-nya tentang BBM, LPG, dan harga. Pertamina tidak dibingkai sebagai agen transisi.

3. **Nikel ambigu** — di satu sisi komoditas masa depan (baterai EV), di sisi lain terkait erat dengan batu bara dan ekstraksi.

4. **Optimisme dari pidato** — narasi positif datang dari pengumuman Prabowo (PLTS 100 GW, KTT ASEAN), bukan laporan realisasi lapangan.

5. **PLN negatif karena krisis teknis** — sentimen buruk bukan dari kebijakan transisi, tapi pemadaman listrik Jakarta (22-24 April).

### Rekomendasi

1. Perluas keyword: energi surya, geothermal, PLTS, PLTA, biomassa, cofiring, CCS, karbon
2. Perpanjang rentang waktu: 6-12 bulan untuk volume data meaningful
3. Kombinasikan scope: channel-level untuk sentimen makro, keyword-filtered untuk presisi
4. Gunakan `entities/{word}/cooccurrence` untuk analisis yang lebih fokus

---
*Dibuat dengan Semantik API. Metodologi: keyword-filtered endpoints + entity-level endpoints.*